`Obligatorio BI - Aurrecochea, Cancela & Esquibel`
### Parte 4 - Consultas SQL y su análisis.
# Análisis de libros de Ciencia Ficción.

### Paso previo: Crear entorno virtual e instalar dependencias

Antes de ejecutar este notebook, correr en la terminal:

`Mac/Linux:`
```bash
python3 -m venv venv_bi
source venv_bi/bin/activate
pip install jupyter mysql-connector-python pandas matplotlib seaborn
```

`Windows:`
```bash
python -m venv venv_bi
venv_bi\Scripts\activate
pip install jupyter mysql-connector-python pandas matplotlib seaborn
```

### Importar librerías y conectar a MySQL

In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='', # contraseña de la DB
    database='obligatorio_bi'
)
print("Conexión exitosa")

Conexión exitosa


`Consulta 1`
### ¿Cuáles son los 10 libros mejor valorados y en qué subgénero SF están?

In [6]:
query1 = """
SELECT 
    dl.book_title,
    da.author_name,
    fm.rating_score,
    fm.rating_votes,
    ds.nombre_subgenero
FROM Fact_Metricas fm
JOIN Dim_Libro dl ON fm.id_libro = dl.id_libro
JOIN Dim_Autor da ON fm.id_autor = da.id_autor
JOIN Bridge_Subgenero bs ON fm.id_libro = bs.id_libro
JOIN Dim_Subgenero ds ON bs.id_subgenero = ds.id_subgenero
WHERE fm.rating_votes >= 1000
ORDER BY fm.rating_score DESC
LIMIT 10
"""

df1 = pd.read_sql(query1, conn)
df1

/var/folders/y8/05j358gs2xg_bd_sfwsvqf8c0000gn/T/ipykernel_84522/143606051.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


,book_title,author_name,rating_score,rating_votes,nombre_subgenero
0,Words of Radiance,Brandon Sanderson,4.74,220750,Apocalyptic
1,The Outlander Series,Diana Gabaldon,4.73,8891,Time Travel
2,Saga: Book One,Brian K. Vaughan,4.68,9472,Space Opera
3,Saga: Book One,Brian K. Vaughan,4.68,9472,Robots
4,天官赐福 [Tiān Guān Cì Fú,Mò Xiāng Tóngxiù,4.68,3777,Alternate universe
5,Transmetropolitan V. 0-10,Warren Ellis,4.67,1879,Cyberpunk
6,Clockwork Angel; Clockwork Prince; Clockwork P...,Cassandra Clare,4.67,14796,Steampunk
7,"Fullmetal Alchemist, Vol. 23",Hiromu Arakawa,4.67,8479,Steampunk
8,"Fullmetal Alchemist, Vol. 26",Hiromu Arakawa,4.67,5542,Steampunk
9,Saga: Book Two,Brian K. Vaughan,4.66,4917,Space Opera


`Análisis consulta 1`

Se aplica un filtro de rating_votes >= 1000 porque sin este filtro 
los primeros resultados mostraban libros con rating perfecto de 5.00 
pero con solo 1, 2 o 4 votos. Un libro con un solo voto de 5 estrellas 
no es estadísticamente representativo de la calidad real del libro.
Al exigir un mínimo de 1000 votos garantizamos que el rating promedio 
refleje la opinión de una cantidad significativa de lectores, haciendo 
el análisis mucho más confiable y útil.

Al establecer el umbral de 1000 votos, los resultados cambian 
drásticamente y se vuelven mucho más informativos. El libro mejor 
valorado es Words of Radiance de Brandon Sanderson con un rating de 4.74 
sobre 220,750 votos, lo que lo convierte en el resultado más confiable 
del ranking por amplio margen. Asimismo, Words of Radience es uno de los mayores bestsellers del género y alcanzó el puesto número 1 en la lista del New York Times en su lanzamiento por lo tanto tiene sentido que predomine en el ranking, información extraida de [Wikipedia](https://es.wikipedia.org/wiki/Palabras_radiantes).

Le sigue The Outlander Series de Diana 
Gabaldon con 4.73 y 8,891 votos, y Saga: Book One de Brian K. Vaughan 
con 4.68 y 9,472 votos.

Un dato interesante es que Saga: Book One aparece dos veces en el 
ranking, asociado a los subgéneros Space Opera y Robots. Esto se debe 
al modelo N:M implementado con Bridge_Subgenero, que permite que un 
mismo libro pertenezca a múltiples subgéneros simultáneamente. Lo mismo 
ocurre con otros títulos del ranking.

En cuanto a la distribución por subgénero, Steampunk es el más 
representado con 3 libros en el top 10, seguido por Space Opera con 2. 
El autor Hiromu Arakawa aparece dos veces con diferentes volúmenes de 
Fullmetal Alchemist, lo que sugiere que las sagas de alta calidad 
mantienen un rating consistente a lo largo de sus entregas.

`Consulta 2`
###  ¿Qué autores tienen el mayor promedio de rating con más de 3 libros?

In [7]:
query2 = """
SELECT 
    da.author_name,
    COUNT(fm.id_libro) AS cantidad_libros,
    ROUND(AVG(fm.rating_score), 2) AS rating_promedio,
    SUM(fm.rating_votes) AS total_votos
FROM Fact_Metricas fm
JOIN Dim_Autor da ON fm.id_autor = da.id_autor
WHERE fm.rating_votes >= 1000
GROUP BY da.author_name
HAVING COUNT(fm.id_libro) > 3
ORDER BY rating_promedio DESC
LIMIT 10
"""

df2 = pd.read_sql(query2, conn)
df2

/var/folders/y8/05j358gs2xg_bd_sfwsvqf8c0000gn/T/ipykernel_84522/2030256381.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


,author_name,cantidad_libros,rating_promedio,total_votos
0,Hiromu Arakawa,26,4.60,365771.0
1,J.K. Rowling,8,4.49,23950035.0
2,Hajime Isayama,12,4.48,278664.0
3,Katsuhiro Otomo,6,4.46,68116.0
4,Sarah Lyons Fleming,6,4.42,18312.0
5,Naoki Urasawa,8,4.41,24938.0
6,Patricia Briggs,4,4.39,156321.0
7,Hailey Turner,6,4.38,11954.0
8,Jodi Taylor,28,4.37,148843.0
9,April White,4,4.37,10665.0


`Análisis consulta 2`

Para esta consulta se mantiene el filtro de rating_votes >= 1000 por 
las mismas razones que en la consulta anterior, y se agrega la condición 
HAVING COUNT(id_libro) > 3 para considerar solo autores con más de 3 
libros en el dataset, garantizando que el promedio sea representativo 
de su obra y no de un único libro excepcional.

El resultado más destacado es Hiromu Arakawa que lidera el ranking con 
un rating promedio de 4.60 sobre 26 libros y 365,771 votos totales. 
Arakawa es la autora de Fullmetal Alchemist, una de las series de manga 
más aclamadas de la historia, lo que explica su alto rating consistente 
a lo largo de múltiples volúmenes.

En segundo lugar aparece J.K. Rowling con 8 libros y un rating promedio 
de 4.49 sobre 23,950,035 votos totales — el mayor volumen de votos del 
ranking por una diferencia abismal, lo que refleja su popularidad 
masiva a nivel mundial. Sin embargo, es importante notar que Rowling 
aparece con solo 8 libros cuando en realidad tiene una obra mucho más 
extensa. Esto se debe a dos razones: el dataset de Kaggle está acotado 
específicamente a libros de ciencia ficción y sus subgéneros, excluyendo 
gran parte de su obra que está catalogada como fantasía, y además cada 
subgénero tiene un límite de libros lo que hace 
que el dataset sea una muestra y no la totalidad de Goodreads.

Hajime Isayama aparece en tercer lugar con 12 libros y 4.48 de promedio, 
siendo el autor de Attack on Titan, otra serie de manga enormemente 
popular. La presencia de tres autores de manga en el top 10 (Arakawa, 
Isayama y Urasawa) es un dato interesante que sugiere que el manga de 
ciencia ficción tiende a mantener ratings muy consistentes a lo largo 
de sus series.

`Consulta 3`
### ¿Qué subgénero de ciencia ficción tiene el mayor rating promedio?

In [10]:
query3 = """
SELECT 
    ds.nombre_subgenero,
    COUNT(fm.id_libro) AS cantidad_libros,
    ROUND(AVG(fm.rating_score), 2) AS rating_promedio,
    ROUND(AVG(fm.rating_votes), 0) AS votos_promedio
FROM Fact_Metricas fm
JOIN Bridge_Subgenero bs ON fm.id_libro = bs.id_libro
JOIN Dim_Subgenero ds ON bs.id_subgenero = ds.id_subgenero
WHERE fm.rating_votes >= 1000
GROUP BY ds.nombre_subgenero
ORDER BY rating_promedio DESC
"""

df3 = pd.read_sql(query3, conn)
df3

/var/folders/y8/05j358gs2xg_bd_sfwsvqf8c0000gn/T/ipykernel_84522/744298936.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


,nombre_subgenero,cantidad_libros,rating_promedio,votos_promedio
0,Military,809,4.08,12567.0
1,Alternate universe,1165,4.05,153667.0
2,Space Opera,1137,4.02,21289.0
3,Aliens,1022,3.99,27336.0
4,Robots,638,3.99,43077.0
5,Hard,850,3.97,52738.0
6,Steampunk,672,3.96,38850.0
7,Cyberpunk,611,3.95,80590.0
8,Time Travel,914,3.95,36322.0
9,Apocalyptic,1119,3.94,74014.0


`Análisis consulta 3`

Military SF lidera el ranking con un rating promedio de 4.08 seguido 
por Alternate Universe con 4.05 y Space Opera con 4.02. Los peores valorados son  Dystopia y Alternate History ambos con 3.91.

Asimismo, se destaca la diferencia mínima entre el subgénero mejor y 
peor valorado, el cual es de apenas 0.17 puntos, lo que sugiere que en general 
todos los subgéneros de ciencia ficción tienen una recepción similar 
por parte de los lectores de Goodreads.

Para complementar el análisis, Alternate universe a parte de ocupar el segundo lugar del ranking con un rating promedio de 4.05, también tiene el mayor promedio de votos por libro, muy por encima del resto. Indicando que los libros de AU aparte de ser bien valorados son populares y leídos por una gran audiencia.
Military lidera en rating pero tiene una cantidad menor de votos, podemos interpretar que es una audiencia más reducida en comparación de AU pero fiel al subgénero, obteniendo altas valoraciones por los lectores.
Cyberpunk es el segundo con más valoraciones pero tiene solo 3.95 de rating promedio. Con Dystopia pasa algo similar, tiene un alto promedio de votos y un rating más bajo. Unos de los subgéneros más leídos en libros de ciencia ficción son Cyberpunk y Dystopia; su popularidad ha ido en aumento por las películas y videojuegos pero las expectativas son tan altas que a veces el libro no las cumple, o parte de los consumidores no son fanáticos de la lectura y no logran continuar con ella a pesar de su gusto por esos subgéneros.

Cyberpunk atrae lectores debido a la popularidad del género, impulsada por películas y videojuegos pero las expectativas son tan altas que a veces el libro no las cumple, o consumidores del género que no leen y se compran los libros por el fin anterior.

Es importante para la interpretación de resultados tener en cuenta la cantidad total de libros y algunos cambios realizados en el ETL. Mientras que Dystopia tiene 1228 libros mientras que Cyberpunk tiene 
solo 611. Esto no significa necesariamente que haya menos libros de 
Cyberpunk en Goodreads, sino que es consecuencia directa de cómo se 
construyó el dataset y cómo se manejaron los duplicados durante el ETL.

El dataset de Kaggle tiene 12 CSVs, uno por subgénero, y muchos libros 
aparecen en más de un CSV simultáneamente porque pertenecen a múltiples 
subgéneros. Durante el proceso de carga en Pentaho esto generaba 
registros duplicados ya que el mismo goodreads_id aparecía múltiples 
veces en el staging. Para resolver esto se tomó la decisión de agrupar 
por goodreads_id en las transformaciones de dimensiones y fact table, 
tomando un único registro por libro. Sin embargo en Bridge_Subgenero 
se mantienen todas las relaciones libro-subgénero, lo que permite que 
un libro cuente para múltiples subgéneros en este análisis.

Adicionalmente, en Bridge_Genero se presentó un problema similar: al 
cargar los datos sin restricciones de clave primaria se generaron 
combinaciones duplicadas de id_libro e id_genero. La solución aplicada 
fue remover temporalmente la primary key y las foreign keys de la tabla, 
cargar todos los datos, ejecutar un DELETE para eliminar los duplicados 
manteniendo el registro con mayor cantidad de votos, y finalmente 
restaurar las restricciones. Todo este proceso fue automatizado dentro 
del Job de Pentaho para garantizar que cada ejecución del ETL produzca 
resultados limpios y consistentes.


